# G1 Academy Bonus - Task 5: services and mode switching

## Introduction
This task builds `notes.txt` section 3 end to end: a safe `damp_mode`, `toggle_service`/`toggle_gait`, and a composite `toggle_custom_mode(mode_name, language, voice, headlight_color)` that announces the mode with Piper, keeps the headlight refreshed for as long as the mode is active, switches the FSM, and cleanly unwinds - thread joined and a safe fallback mode - on exit.

In [ ]:
import time
from unitree_sdk2py.core.channel import ChannelFactoryInitialize, ChannelPublisher, ChannelSubscriber
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowState_

import sys
if ".." not in sys.path:
    sys.path.append("..")
from sdk_wrapper import ensure_channel_factory

ensure_channel_factory(0, "eth0")

class Latest:
    def __init__(self, topic, message_type, queue_len=10):
        self.message = None
        self.timestamp = 0.0
        self.subscriber = ChannelSubscriber(topic, message_type)
        self.subscriber.Init(self._callback, queue_len)
    def _callback(self, message):
        self.message = message
        self.timestamp = time.time()
    def fresh(self, max_age_s=0.5):
        return self.message is not None and time.time() - self.timestamp <= max_age_s

lowstate_sub = Latest("rt/lowstate", LowState_)

## Task 1 - Native mode/service clients + `damp_mode()`
`LocoClient` changes locomotion FSM state; `MotionSwitcherClient` checks/releases the higher-level motion owner; `RobotStateClient` lists/switches services. Do not invent FSM IDs - reuse the documented `FSM_IDS` map. `damp` (id `1`) is the always-available safe fallback: bounded joint damping, no locomotion - call it before an emergency stop or before releasing controller ownership of any other publisher.

In [ ]:
from unitree_sdk2py.g1.loco.g1_loco_client import LocoClient
from unitree_sdk2py.comm.motion_switcher.motion_switcher_client import MotionSwitcherClient
try:
    from unitree_sdk2py.b2.robot_state.robot_state_client import RobotStateClient
except ImportError:
    from unitree_sdk2py.go2.robot_state.robot_state_client import RobotStateClient

FSM_IDS = {"zero_torque": 0, "damp": 1, "prepare": 4, "walk": 501, "run": 802}

loco = LocoClient(); loco.SetTimeout(5.0); loco.Init()
switcher = MotionSwitcherClient(); switcher.SetTimeout(5.0); switcher.Init()
robot_state_client = RobotStateClient(); robot_state_client.SetTimeout(5.0); robot_state_client.Init()

def damp_mode():
    result = loco.SetFsmId(FSM_IDS["damp"])
    return 0 if result is None else int(result)

#damp_mode()

## Task 2 - `toggle_service(name)` and `toggle_gait()`
`toggle_service` flips a named `RobotStateClient` service on/off based on its currently reported status. `toggle_gait` prefers `SetBalanceMode`/`SetGaitType` where the installed SDK exposes them, and falls back to `BalanceStand`/`SetFsmId("walk")` to return to the default gait; it always inspects the return code instead of assuming success.

In [ ]:
def rpc_code(value):
    return 0 if value is None else int(value)

def get_service(name):
    code, rows = robot_state_client.ServiceList()
    if int(code) != 0:
        raise RuntimeError(f"ServiceList failed: {code}")
    target = str(name).strip().lower()
    return next((r for r in rows if str(r.name).strip().lower() == target), None)

def set_service(name, enabled):
    row = get_service(name)
    if row is None:
        raise ValueError(f"Unknown service: {name}")
    code = rpc_code(robot_state_client.ServiceSwitch(row.name, bool(enabled)))
    return {"name": row.name, "previous_status": int(row.status), "enabled": bool(enabled), "code": code}

def toggle_service(name):
    row = get_service(name)
    if row is None:
        raise ValueError(f"Unknown service: {name}")
    enable = int(row.status) != 0  # status 0 is ON, status 1 is OFF
    return set_service(row.name, enable)

_gait_override = None
def toggle_gait():
    global _gait_override
    target = 0 if (_gait_override or 0) else 1
    codes = []
    if target:
        for method_name in ("SetBalanceMode", "SetGaitType"):
            if hasattr(loco, method_name):
                try:
                    code = rpc_code(getattr(loco, method_name)(1))
                except Exception as exc:
                    codes.append((method_name, type(exc).__name__, str(exc)))
                    continue
                codes.append((method_name, code))
                if code == 0:
                    _gait_override = 1
                    return {"gait": 1, "codes": codes}
    else:
        if hasattr(loco, "BalanceStand"):
            try:
                codes.append(("BalanceStand", rpc_code(loco.BalanceStand(0))))
            except Exception as exc:
                codes.append(("BalanceStand", type(exc).__name__, str(exc)))
        for method_name in ("SetBalanceMode", "SetGaitType"):
            if hasattr(loco, method_name):
                try:
                    codes.append((method_name, rpc_code(getattr(loco, method_name)(0))))
                except Exception as exc:
                    codes.append((method_name, type(exc).__name__, str(exc)))
        if hasattr(loco, "SetFsmId"):
            try:
                codes.append(("SetFsmId", rpc_code(loco.SetFsmId(FSM_IDS["walk"]))))
            except Exception as exc:
                codes.append(("SetFsmId", type(exc).__name__, str(exc)))
        if any(len(item) == 2 and item[1] == 0 for item in codes):
            _gait_override = 0
    return {"gait": target, "codes": codes}


In [ ]:
toggle_service("vui_service")
#toggle_service("ai_sport")  # motion service; only toggle when you intentionally want to release/re-enable it


In [ ]:
toggle_gait()

## Task 3 - `say`/`set_headlight` (compact reuse)
Task 3 develops these two helpers in full; they are reproduced compactly here because `toggle_custom_mode` below depends on both.

In [ ]:
import re, sys
sys.path.append("..")
import threading
from unitree_sdk2py.g1.audio.g1_audio_client import AudioClient
from util import play_piper_text

audio_client = AudioClient(); audio_client.SetTimeout(5.0); audio_client.Init()

def say(text, language="en", volume=100, voice=None):
    # voice overrides the language-derived Piper voice model name when the installed voice
    # differs from util.PIPER_VOICES's default for that language.
    if voice:
        import util
        util.PIPER_VOICES[str(language).lower()] = voice
    return play_piper_text(audio_client, text, language=language, volume=volume)

_NAMED_COLORS = {"white": (255, 255, 255), "red": (255, 0, 0), "green": (0, 255, 0), "blue": (0, 0, 255),
                  "yellow": (255, 255, 0), "cyan": (0, 255, 255), "magenta": (255, 0, 255)}
def parse_color(value):
    if isinstance(value, tuple):
        return value
    value = str(value).strip().lower()
    if value in _NAMED_COLORS:
        return _NAMED_COLORS[value]
    if re.fullmatch(r"#?[0-9a-fA-F]{6}", value):
        value = value.lstrip("#")
        return tuple(int(value[i:i + 2], 16) for i in (0, 2, 4))
    raise ValueError("color must be a name or #RRGGBB")

def led_control_was_accepted(code):
    return int(code) in (0, 3104)

def set_headlight_once(rgb):
    return int(audio_client.LedControl(*rgb))

## Task 4 - `toggle_custom_mode(mode_name, language, voice, headlight_color)`
First call for a given `mode_name` announces it with Piper, switches the FSM, and starts a background thread that keeps refreshing the requested headlight color every 0.2s. Calling it again with the same active mode name (or any other name) cleanly exits: signal the thread, join it, turn the light off, and fall back to `damp_mode()` for a safe state transition.

In [ ]:
_custom_modes = {
    "greet": {"fsm": "prepare", "announce": "Entering greet mode."},
    "patrol": {"fsm": "walk", "announce": "Entering patrol mode."},
}
_custom_mode_state = {"active": None, "stop": None, "thread": None}

def _exit_custom_mode():
    state = _custom_mode_state
    if state["thread"] is not None:
        state["stop"].set()
        state["thread"].join()
    left = state["active"]
    damp_mode()
    state.update(active=None, stop=None, thread=None)
    return {"exited": left}

def toggle_custom_mode(mode_name, language="en", voice=None, headlight_color="green"):
    if _custom_mode_state["active"] == mode_name:
        return _exit_custom_mode()
    if _custom_mode_state["active"] is not None:
        _exit_custom_mode()
    spec = _custom_modes[mode_name]
    say(spec["announce"], language=language, voice=voice)
    fsm_code = rpc_code(loco.SetFsmId(FSM_IDS[spec["fsm"]]))
    rgb = parse_color(headlight_color)
    stop_event = threading.Event()
    def worker():
        while not stop_event.is_set():
            code = set_headlight_once(rgb)
            if not led_control_was_accepted(code):
                break
            stop_event.wait(0.2)
        set_headlight_once((0, 0, 0))
    thread = threading.Thread(target=worker, daemon=False)
    thread.start()
    _custom_mode_state.update(active=mode_name, stop=stop_event, thread=thread)
    return {"mode": mode_name, "fsm_code": fsm_code}

toggle_custom_mode("greet", language="en", headlight_color="yellow")  # enter

In [ ]:
toggle_custom_mode("greet")  # exit: stops the thread and calls damp_mode()

### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.